# Network Intrusion · редкие атаки и экстремальный дисбаланс (Spark, 2.8 млн сетевых потоков)

**Данные:** CICIDS2017 (Canadian Institute for Cybersecurity) — сетевые потоки рабочей недели: ~2.8 млн строк, 78 числовых признаков
потока (длительности, размеры пакетов, флаги TCP, скорости…) и метка трафика. Трафик записан на тестовом стенде реальным захватом пакетов
и размечен по расписанию атак; это не сгенерированные моделью строки. Kaggle: `chethuhn/network-intrusion-dataset`.

**Задача карточки:** найти **редкие и скрытные** атаки — Bot, Web Attack (Brute Force / XSS / SQL Injection), Infiltration, Heartbleed —
среди всего остального трафика (нормальный + «громкие» DoS/DDoS/PortScan/Patator, которые заметны и без модели).
Положительный класс — доли процента данных: именно в такой постановке accuracy и ROC-AUC перестают быть информативными.

**Что сравниваем — способы борьбы с дисбалансом на Spark ML:**
`RandomForest` без весов · `RandomForest` с весами классов · `RandomForest` + undersampling 20:1 · `GBT` с весами · `LogReg` с весами.

**Метрики:** MCC (hero), PR-AUC, recall редких семейств атак и доля ложных тревог по семействам.

Честная оговорка: разбиение случайное (после удаления точных дубликатов), поэтому оценка оптимистична относительно реального развёртывания
(атаки одного семейства записаны подряд на стенде и похожи друг на друга). Для сравнения методов между собой этого достаточно.

In [ ]:
import os, re, sys, json, math, time, socket, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / 'requirements.txt').exists() and (parent / 'notebooks').exists():
            return parent
    raise RuntimeError('Корень проекта не найден')


SEED    = 42
PROJECT = find_project_root()
TASK    = 'network_intrusion'

MODEL_DIR = PROJECT / 'models' / 'classification' / TASK
ART_DIR   = PROJECT / 'artifacts' / 'classification' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT  :', PROJECT)
print('MODEL_DIR:', MODEL_DIR)
print('ART_DIR  :', ART_DIR)

## 1. Метрики без Spark (numpy)

Spark считает гистограмму скоров (1001 бин × 2 класса) на валидации и тесте; из неё на драйвере получаем confusion matrix при любом пороге,
кривые ROC/PR, порог по максимуму MCC и все метрики. Функции сверены с `sklearn` (расхождение ~1e-16).

In [ ]:
# ---------- метрики без Spark: считаются по гистограммам / confusion matrix ----------
import math, json
import numpy as np

NBINS = 1000          # разрешение порога = 0.001


def _div(a, b):
    return a / b if b else 0.0


def _fbeta(p, r, beta):
    d = beta * beta * p + r
    return (1 + beta * beta) * p * r / d if d else 0.0


def binary_metrics(tp, fp, fn, tn):
    """Все метрики бинарной классификации из четырёх чисел confusion matrix."""
    tp, fp, fn, tn = (float(x) for x in (tp, fp, fn, tn))
    n, P, N = tp + fp + fn + tn, tp + fn, tn + fp
    prec_p, rec_p = _div(tp, tp + fp), _div(tp, P)
    prec_n, rec_n = _div(tn, tn + fn), _div(tn, N)
    f1p, f05p, f2p = (_fbeta(prec_p, rec_p, b) for b in (1.0, 0.5, 2.0))
    f1n, f05n, f2n = (_fbeta(prec_n, rec_n, b) for b in (1.0, 0.5, 2.0))
    acc = _div(tp + tn, n)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _div(tp * tn - fp * fn, den)
    pe = _div((tp + fp) * P + (tn + fn) * N, n * n)
    kappa = _div(acc - pe, 1 - pe)
    return {
        'accuracy': acc,
        'balanced_accuracy': (rec_p + rec_n) / 2,
        'precision_macro': (prec_p + prec_n) / 2,
        'recall_macro': (rec_p + rec_n) / 2,
        'f1_macro': (f1p + f1n) / 2,
        'f1_weighted': _div(f1p * P + f1n * N, n),
        'f0.5_macro': (f05p + f05n) / 2,
        'f2_macro': (f2p + f2n) / 2,
        'mcc': mcc,
        'kappa': kappa,
        'specificity': rec_n,
        'sensitivity': rec_p,
        'precision_pos': prec_p,
        'recall_pos': rec_p,
        'f1_pos': f1p,
        'f0.5_pos': f05p,
        'f2_pos': f2p,
    }


def hist_counts(neg, pos):
    """neg/pos — гистограммы длины NBINS+1 (бин = floor(score*NBINS)).
    Для порога k/NBINS (k = 0..NBINS) объект предсказан положительным при score >= k/NBINS.
    Возвращает массивы tp, fp, fn, tn длины NBINS+1."""
    neg = np.asarray(neg, dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    neg_ge = np.cumsum(neg[::-1])[::-1]
    pos_ge = np.cumsum(pos[::-1])[::-1]
    P, N = pos.sum(), neg.sum()
    return pos_ge, neg_ge, P - pos_ge, N - neg_ge          # tp, fp, fn, tn


def _mcc_vec(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=den > 0)


def best_threshold_k(neg, pos, objective='mcc', cost_fp=1.0, cost_fn=1.0):
    """Индекс k лучшего порога на (валидационной) гистограмме.
    objective: 'mcc' (максимум) | 'cost' (минимум fp*cost_fp + fn*cost_fn)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    if objective == 'mcc':
        return int(np.argmax(_mcc_vec(tp, fp, fn, tn)))
    if objective == 'cost':
        return int(np.argmin(fp * cost_fp + fn * cost_fn))
    raise ValueError(objective)


def sweep_block(neg, pos, n_points=101):
    """Подвыборка порогов 0..1 для фронта: счётчики tp/fp/fn/tn на тестовой гистограмме."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    ks = np.unique(np.round(np.linspace(0, NBINS, n_points)).astype(int))
    return {
        'thresholds': [round(float(k) / NBINS, 4) for k in ks],
        'tp': [int(tp[k]) for k in ks], 'fp': [int(fp[k]) for k in ks],
        'fn': [int(fn[k]) for k in ks], 'tn': [int(tn[k]) for k in ks],
    }


def _thin(x, y):
    out_x, out_y = [], []
    for a, b in zip(x, y):
        if out_x and abs(a - out_x[-1]) < 2e-3 and abs(b - out_y[-1]) < 2e-3:
            continue
        out_x.append(round(float(a), 4)); out_y.append(round(float(b), 4))
    return out_x, out_y


def _cap(x, y, n=140):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).round().astype(int))
    return [x[i] for i in idx], [y[i] for i in idx]


def curves_block(neg, pos):
    """ROC и PR-кривые по гистограмме (для графиков; точные AUC считает Spark)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P, N = max(tp[0], 1.0), max(fp[0], 1.0)
    tpr, fpr = tp / P, fp / N
    prec = np.divide(tp, tp + fp, out=np.ones_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    fx, fy = _cap(*_thin(fpr[::-1], tpr[::-1]))
    rx, ry = _cap(*_thin(tpr[::-1], prec[::-1]))
    return {'roc': {'fpr': fx, 'tpr': fy}, 'pr': {'recall': rx, 'precision': ry}}


def multiclass_metrics(cm):
    """cm[i][j] — сколько объектов истинного класса i предсказано как j."""
    cm = np.asarray(cm, dtype=np.float64)
    K, s = cm.shape[0], cm.sum()
    t, p, d = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    prec = np.divide(d, p, out=np.zeros(K), where=p > 0)
    rec = np.divide(d, t, out=np.zeros(K), where=t > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    acc = d.sum() / s
    mcc_den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0))
    mcc = (d.sum() * s - (p * t).sum()) / mcc_den if mcc_den else 0.0
    pe = (p * t).sum() / (s * s)
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    idx = np.arange(K)
    w = (idx[:, None] - idx[None, :]) ** 2 / max((K - 1) ** 2, 1)
    expected = np.outer(t, p) / s
    qwk = 1 - (w * cm).sum() / (w * expected).sum() if (w * expected).sum() else 0.0
    return {
        'accuracy': acc, 'balanced_accuracy': rec.mean(),
        'precision_macro': prec.mean(), 'recall_macro': rec.mean(), 'f1_macro': f1.mean(),
        'precision_weighted': (prec * t).sum() / s, 'recall_weighted': (rec * t).sum() / s,
        'f1_weighted': (f1 * t).sum() / s, 'mcc': mcc, 'kappa': kappa, 'qwk': qwk,
        '_per_class': {'precision': prec, 'recall': rec, 'f1': f1, 'support': t},
    }


def to_native(o):
    """Рекурсивно приводит numpy/NaN/Inf к JSON-совместимым значениям (NaN → null)."""
    if isinstance(o, dict):
        return {str(k): to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_native(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_native(o.tolist())
    if isinstance(o, np.generic):
        return to_native(o.item())
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else o
    if hasattr(o, '__fspath__'):
        return str(o)
    return o


def dump_json(obj, path):
    path.write_text(json.dumps(to_native(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

## 2. Spark: сессия, ресурсы, общий каталог

Executor: 8 ядер / 10 ГБ, драйвер 2 ГБ. Без Python-UDF — только Spark SQL и Spark ML.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
import requests

# ---- бюджет ресурсов берём из docker-compose ----
# spark-worker:  command --cores 8 --memory 10g  (limit cpus=8, memory=12g)
# ml (driver):   лимит cpus=3.0, memory=6g       → драйвер 2g, локально не считаем
SPARK_MASTER_URL = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
CORES      = int(os.environ.get('SPARK_CORES', os.environ.get('WEATHER_SPARK_CORES', '8')))                 # worker: --cores 8
EXEC_MEM   = os.environ.get('SPARK_EXECUTOR_MEMORY', os.environ.get('WEATHER_SPARK_EXECUTOR_MEMORY', '8g'))  # worker: --memory 10g, limit 12g
DRIVER_MEM = os.environ.get('SPARK_DRIVER_MEMORY', '2g')       # контейнер ml ограничен 6g — драйвер держим скромным
LOCAL_CORES = int(os.environ.get('SPARK_LOCAL_CORES', '3'))    # контейнер ml: cpus 3.0 (только для SPARK_MODE=local)
FORCE_MODE = os.environ.get('SPARK_MODE', 'cluster')           # cluster (по умолчанию) | auto | local
ALLOW_LOCAL_FALLBACK = os.environ.get('ALLOW_LOCAL_FALLBACK', '0') == '1'
EVENT_DIR  = Path('/opt/spark/events')                         # общий каталог для Spark History Server


def _master_tcp_ok(url, timeout=2.0):
    try:
        host, port = url.split('://', 1)[1].split(':')
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except Exception:
        return False


def _alive_workers(url):
    """Список живых worker из Web UI мастера (8085). None — UI недоступен."""
    try:
        host = url.split('://', 1)[1].split(':')[0]
        info = requests.get(f'http://{host}:8085/json/', timeout=3).json()
        return [w for w in info.get('workers', []) if w.get('state') == 'ALIVE']
    except Exception:
        return None


def choose_mode():
    """
    cluster (default) — требуем живой spark-master с ALIVE-worker. Нет → RuntimeError.
    auto             — пробуем cluster, при неудаче fallback на local (с громким предупреждением).
    local            — принудительно local[3] (для отладки; урезает данные, ест 3 ядра ml).
    ALLOW_LOCAL_FALLBACK=1 дополнительно разрешает fallback в режиме cluster.
    """
    if FORCE_MODE == 'local':
        print(f'⚠ SPARK_MODE=local: работаем на {LOCAL_CORES} ядрах ml (данные урежутся, только для отладки)')
        return 'local'

    if FORCE_MODE not in ('cluster', 'auto'):
        raise ValueError(f'SPARK_MODE={FORCE_MODE!r} (ожидалось auto|cluster|local)')

    if _master_tcp_ok(SPARK_MASTER_URL):
        workers = _alive_workers(SPARK_MASTER_URL)
        if workers is None or len(workers) > 0:
            return 'cluster'
        print('⚠ spark-master отвечает, но ALIVE-worker нет — задачи выполнять некому')

    if FORCE_MODE == 'auto' or ALLOW_LOCAL_FALLBACK:
        print(f'⚠ падаю в fallback local[{LOCAL_CORES}] — это НЕ полный прогон, 3 ядра ml, DATA_FRACTION=0.15')
        return 'local'

    raise RuntimeError(
        'Spark-кластер недоступен, а SPARK_MODE=cluster (по умолчанию).\n'
        f'  master: {SPARK_MASTER_URL}\n'
        '  Запустите профиль:\n'
        '      docker compose --profile spark up -d\n'
        '  Проверить: http://localhost:8085 (master UI), http://localhost:8086 (worker UI)\n'
        '  Для локальной отладки без кластера:\n'
        '      SPARK_MODE=local     (урежет данные до 15%, 3 ядра ml)\n'
        '      SPARK_MODE=auto      (попробует cluster, потом local)\n'
    )


def start_spark(mode):
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    b = (SparkSession.builder.appName(f'ml-playground · {TASK}')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.driver.maxResultSize', '512m'))
    if mode == 'cluster':
        b = (b.master(SPARK_MASTER_URL)
              .config('spark.submit.deployMode', 'client')     # driver — здесь, executor — на worker
              .config('spark.driver.memory', DRIVER_MEM)
              .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))
              .config('spark.driver.bindAddress', '0.0.0.0')
              .config('spark.driver.port', '7078')
              .config('spark.blockManager.port', '7079')
              .config('spark.executor.memory', EXEC_MEM)       # 8g
              .config('spark.executor.cores', str(CORES))      # 8
              .config('spark.cores.max', str(CORES))           # 8 → 1 executor × 8 cores на worker
              .config('spark.sql.shuffle.partitions', str(CORES * 6))   # 48
              .config('spark.default.parallelism', str(CORES * 6)))     # 48
    else:
        b = (b.master(f'local[{LOCAL_CORES}]')
              .config('spark.driver.memory', '4g')
              .config('spark.sql.shuffle.partitions', str(LOCAL_CORES * 4)))
    if EVENT_DIR.is_dir():
        b = b.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', f'file://{EVENT_DIR}')
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark


# Каталог, который видят И драйвер (ml), И executor (spark-worker). Порядок кандидатов:
#  1) /spark-data              — общий docker-volume (см. патч docker-compose в README)
#  2) <project>/data/spark_cache — работает, если в worker смонтирован ./data
#  3) <project>/models/.spark_cache — ./models уже смонтирован в ml и в spark-worker по одному пути (/app/models)
SHARED_CANDIDATES = [Path('/spark-data'), PROJECT / 'data' / 'spark_cache', PROJECT / 'models' / '.spark_cache']


def pick_shared_dir(spark, mode):
    if mode == 'local':
        d = PROJECT / 'data' / 'spark_cache'
        d.mkdir(parents=True, exist_ok=True)
        return d
    for d in SHARED_CANDIDATES:
        try:
            d.mkdir(parents=True, exist_ok=True)
            probe = d / '_probe.txt'
            probe.write_text('ok')
            got = spark.read.text(f'file://{probe}').collect()        # читает executor, не драйвер
            if got and got[0][0] == 'ok':
                return d
        except Exception as e:
            print(f'  {d}: executor не видят каталог ({type(e).__name__})')
    return None


MODE  = choose_mode()
spark = start_spark(MODE)
SHARED = pick_shared_dir(spark, MODE)

# fallback в local разрешён только через choose_mode (по SPARK_MODE/ALLOW_LOCAL_FALLBACK)
# если сюда попали с MODE=cluster и SHARED=None — это проблема монтирования, а не режима
if SHARED is None:
    raise RuntimeError(
        'Ни один общий каталог не виден executor-ам. Проверьте docker-compose:\n'
        '  spark-worker должен монтировать ./data или ./models по тому же пути, что и ml.\n'
        '  Кандидаты: ' + ', '.join(str(p) for p in SHARED_CANDIDATES)
    )

sc = spark.sparkContext
DATA_FRACTION = float(os.environ.get('DATA_FRACTION', '1.0' if MODE == 'cluster' else '0.15'))

SPARK_INFO = {
    'mode': MODE,
    'master': SPARK_MASTER_URL if MODE == 'cluster' else f'local[{LOCAL_CORES}]',
    'spark_version': spark.version,
    'app_id': sc.applicationId,
    'cores': CORES if MODE == 'cluster' else LOCAL_CORES,
    'executor_memory': EXEC_MEM if MODE == 'cluster' else '4g (driver)',
    'default_parallelism': int(sc.defaultParallelism),
    'shuffle_partitions': int(spark.conf.get('spark.sql.shuffle.partitions')),
    'shared_dir': str(SHARED),
    'data_fraction': DATA_FRACTION,
}
print(json.dumps(SPARK_INFO, indent=2, ensure_ascii=False))
if MODE == 'local':
    print(f'\n⚠ local-режим: данные урезаны до {DATA_FRACTION:.0%} (DATA_FRACTION), используются {LOCAL_CORES} ядра ml. '
          'Для полного объёма на 8 ядрах spark-worker: docker compose --profile spark up -d и SPARK_MODE=cluster.')


# ---------- утилиты ----------
TIMINGS = {}


class stage:
    """with stage('имя'): ...  — печатает и копит время этапов (идёт в report.json)."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f'▶ {self.name} ...')
        return self

    def __exit__(self, *exc):
        TIMINGS[self.name] = round(time.time() - self.t, 1)
        print(f'✔ {self.name}: {TIMINGS[self.name]} с')


def ensure_shared_dir(src):
    """Если данные лежат вне общего каталога — копируем, чтобы их увидели executor."""
    src = Path(src).resolve()
    if str(src).startswith(str(SHARED.resolve())):
        return src
    dst = SHARED / 'raw' / src.name
    print(f'копирую {src} → {dst} (чтобы executor увидели файлы)')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return dst


def spark_path(p):
    return f'file://{p}'

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def score_sdf(model, sdf, extra=()):
    """Предсказания модели → DataFrame(label, *extra, score = P(класс 1))."""
    return model.transform(sdf).select('label', *extra, vector_to_array('probability')[1].alias('score'))


def score_histogram(sdf):
    """Гистограмма score по NBINS+1 бинам отдельно для классов 0 и 1 — один проход по данным.
    Из неё без повторных заданий получаем confusion matrix при ЛЮБОМ пороге k/NBINS."""
    b = F.greatest(F.lit(0), F.least(F.floor(F.col('score') * NBINS).cast('int'), F.lit(NBINS)))
    rows = sdf.select(b.alias('b'), F.col('label').cast('int').alias('y')).groupBy('b', 'y').count().collect()
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    for r in rows:
        (pos if r['y'] == 1 else neg)[r['b']] = r['count']
    return neg, pos


def spark_auc(sdf):
    """Точные ROC-AUC и PR-AUC (без даунсэмплинга кривых) средствами Spark."""
    out = {}
    for metric, key in (('areaUnderROC', 'roc_auc'), ('areaUnderPR', 'pr_auc')):
        try:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric, numBins=0)
        except TypeError:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric)
        out[key] = float(ev.evaluate(sdf))
    return out


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])

## 3. Загрузка и подготовка данных

Нюансы CICIDS2017, которые обрабатываем явно:

* в заголовках лишние пробелы и один дублирующийся столбец (`Fwd Header Length` встречается дважды) — нормализуем имена позиционно;
* `Flow Bytes/s` и `Flow Packets/s` содержат `Infinity` и `NaN` (нулевая длительность потока) — заменяем на 0;
* в названиях веб-атак «битый» символ — приводим метки к ASCII;
* в данных много **точных дубликатов** — удаляем до разбиения, иначе одинаковые строки попадут и в train, и в test.

Положительный класс = «редкая атака» (`Bot`, `Web Attack *`, `Infiltration`, `Heartbleed`). Исходное семейство сохраняем — оно нужно для разбора ошибок.

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = str(SHARED / 'kagglehub')
import kagglehub

HANDLES = [os.environ['CICIDS_KAGGLE_HANDLE']] if os.environ.get('CICIDS_KAGGLE_HANDLE') else ['chethuhn/network-intrusion-dataset']
DATA_DIR, DATASET_NAME = None, None
with stage('Загрузка CICIDS2017'):
    for h in HANDLES:
        try:
            DATA_DIR, DATASET_NAME = Path(kagglehub.dataset_download(h)), h
            break
        except Exception as e:
            print(f'  {h}: недоступен ({type(e).__name__}: {str(e)[:120]})')
if DATA_DIR is None:
    DATA_DIR, DATASET_NAME = PROJECT / 'data' / 'cicids2017', 'local: data/cicids2017'
    print('использую локальную копию:', DATA_DIR)

DATA_DIR = ensure_shared_dir(DATA_DIR)
csv_files = sorted(DATA_DIR.rglob('*.csv'))
assert csv_files, f'CSV не найдены в {DATA_DIR}. Положите файлы CICIDS2017 в data/cicids2017/ или задайте CICIDS_KAGGLE_HANDLE.'
print(f'CSV-файлов: {len(csv_files)}, суммарно {sum(p.stat().st_size for p in csv_files) / 1e6:.0f} МБ')
for p in csv_files:
    print('  ', p.name)

raw = spark.read.option('header', True).csv([spark_path(p) for p in csv_files])


def clean_name(c):
    return re.sub(r'[^0-9a-zA-Z]+', '_', c.strip()).strip('_').lower()


names, seen = [], {}
for c in raw.columns:
    n = clean_name(c)
    k = seen.get(n, 0)
    seen[n] = k + 1
    names.append(n if k == 0 else f'{n}_{k}')
raw = raw.toDF(*names)
assert 'label' in names, f'столбец Label не найден, есть: {names[:5]}…{names[-3:]}'
FEATURES = [c for c in names if c != 'label']
print('признаков:', len(FEATURES))

In [ ]:
def finite0(c):
    x = F.col(c).cast('double')
    return F.when(x.isNull() | F.isnan(x) | (F.abs(x) == F.lit(float('inf'))), F.lit(0.0)).otherwise(x).alias(c)


family = F.trim(F.regexp_replace(F.regexp_replace(F.col('label'), '[^A-Za-z0-9]+', ' '), ' +', ' '))
typed = raw.where(F.col('label').isNotNull()).select(*[finite0(c) for c in FEATURES], family.alias('family'))

RARE_LOWER = ['bot', 'infiltration', 'heartbleed']
is_rare = F.lower(F.col('family')).isin(RARE_LOWER) | F.lower(F.col('family')).startswith('web attack')

with stage('Подсчёт строк в исходных CSV'):
    N_RAW = typed.count()
print(f'строк в CSV: {N_RAW:,}')

prepared = (typed.dropDuplicates(FEATURES + ['family'])
            .withColumn('label', is_rare.cast('double'))
            .withColumn('u', F.rand(SEED))
            .withColumn('fold', F.when(F.col('u') < 0.70, 'train').when(F.col('u') < 0.80, 'val').otherwise('test'))
            .drop('u'))
if DATA_FRACTION < 1.0:
    prepared = prepared.sample(False, DATA_FRACTION, SEED)

CLEAN_PATH = SHARED / 'intrusion_clean.parquet'
with stage('Очистка, дедупликация, фолды, запись parquet'):
    prepared.write.mode('overwrite').parquet(spark_path(CLEAN_PATH))     # fold фиксируется в файле: rand() больше не пересчитывается

df = spark.read.parquet(spark_path(CLEAN_PATH))
N_USED = df.count()
print(f'после удаления дубликатов: {N_USED:,} ({N_USED / N_RAW:.1%} от CSV) · партиций: {df.rdd.getNumPartitions()}')

## 4. EDA: насколько всё плохо с балансом

In [ ]:
with stage('EDA-агрегаты'):
    fam_rows = df.groupBy('family').agg(F.count('*').alias('n'), F.max('label').alias('rare')).orderBy(F.desc('n')).collect()
    fold_rows = df.groupBy('fold').agg(F.count('*').alias('n'), F.sum('label').alias('pos')).collect()

FAMILIES = [{'family': r['family'], 'n': int(r['n']), 'is_rare': bool(r['rare'] == 1.0)} for r in fam_rows]
assert any(f['is_rare'] for f in FAMILIES), f'не найдено редких семейств; метки: {[f["family"] for f in FAMILIES]}'
FOLDS = {r['fold']: {'n': int(r['n']), 'pos': int(r['pos']), 'prevalence': float(r['pos']) / int(r['n'])} for r in fold_rows}
N_POS_ALL = sum(v['pos'] for v in FOLDS.values())
PREVALENCE = N_POS_ALL / N_USED
IMBALANCE = (N_USED - N_POS_ALL) / max(N_POS_ALL, 1)
print(f'редкие атаки: {N_POS_ALL:,} из {N_USED:,} → {PREVALENCE:.4%}  (дисбаланс ≈ 1 : {IMBALANCE:,.0f})')
for f in FAMILIES:
    print(f'  {"★" if f["is_rare"] else " "} {f["family"]:28s} {f["n"]:>10,}')
for k, v in FOLDS.items():
    print(f'  fold={k:5s} n={v["n"]:>9,}  редких={v["pos"]:>5,}  prevalence={v["prevalence"]:.4%}')

fig, ax = plt.subplots(figsize=(11, 5))
fd = pd.DataFrame(FAMILIES)
ax.barh(fd.family, fd.n, color=['#f43f5e' if r else '#34d399' for r in fd.is_rare])
ax.set_xscale('log'); ax.invert_yaxis(); ax.set_xlabel('потоков (лог-шкала)')
ax.set_title(f'Семейства трафика · красным — «редкие атаки» ({PREVALENCE:.3%} данных)')
plt.tight_layout(); plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight'); plt.show()

## 5. Подготовка обучающих выборок

Три способа работы с дисбалансом — все на стороне Spark, без копирования данных на драйвер:

1. **Без коррекции** — модель видит естественную долю редких атак.
2. **Веса классов** (`weightCol`): вес положительного класса = `N_neg / N_pos` по train, отрицательного = 1.
3. **Undersampling** (`sampleBy`): отрицательные объекты прореживаются до соотношения 20 : 1; положительные остаются все.

Веса и undersampling смещают вероятности вверх, поэтому порог 0.5 для них бессмыслен — порог всегда подбирается на валидации по максимуму MCC.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Summarizer

UNDER_RATIO = 20

with stage('Векторизация и кэш'):
    vec = (VectorAssembler(inputCols=FEATURES, outputCol='features')
           .transform(df).select('label', 'fold', 'family', 'features'))
    train_cnt = {r['label']: r['count'] for r in vec.where(F.col('fold') == 'train').groupBy('label').count().collect()}
    N_NEG_TR, N_POS_TR = int(train_cnt.get(0.0, 0)), int(train_cnt.get(1.0, 0))
    W_POS = N_NEG_TR / max(N_POS_TR, 1)
    vec = vec.withColumn('w', F.when(F.col('label') == 1.0, F.lit(W_POS)).otherwise(F.lit(1.0)))

    train = vec.where(F.col('fold') == 'train').persist(StorageLevel.MEMORY_AND_DISK)
    val   = vec.where(F.col('fold') == 'val').persist(StorageLevel.MEMORY_AND_DISK)
    test  = vec.where(F.col('fold') == 'test').persist(StorageLevel.MEMORY_AND_DISK)
    N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()

    keep_neg = min(1.0, UNDER_RATIO * N_POS_TR / max(N_NEG_TR, 1))
    train_under = train.sampleBy('label', {0.0: keep_neg, 1.0: 1.0}, SEED).persist(StorageLevel.MEMORY_AND_DISK)
    N_UNDER = train_under.count()

STD = np.array(train.select(Summarizer.metrics('std').summary(F.col('features')).alias('s')).first()['s']['std'].toArray())
print(f'train={N_TRAIN:,} (редких {N_POS_TR:,})  val={N_VAL:,}  test={N_TEST:,}')
print(f'вес редкого класса = {W_POS:,.0f} · undersampling оставляет {keep_neg:.2%} «нормального» трафика → train_under = {N_UNDER:,}')

## 6. Модели

Порог каждой модели подбирается на **валидации** по максимуму MCC; все метрики — на **тесте**.
Для контраста сохраняем и метрики при «наивном» пороге 0.5 (`fixed_05`): разрыв показывает, сколько даёт настройка порога.

Для каждой модели считаем, какая доля потоков **каждого семейства** помечена как атака: у редких семейств это recall,
у остальных — доля ложных тревог. Так видно, *какие* атаки модель реально ловит, а не только итоговый MCC.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier


def family_table(test_scored, k):
    thr = k / NBINS
    rows = (test_scored.groupBy('family')
            .agg(F.count('*').alias('n'), F.sum((F.col('score') >= thr).cast('int')).alias('flagged')).collect())
    rare = {f['family']: f['is_rare'] for f in FAMILIES}
    out = [{'family': r['family'], 'n': int(r['n']), 'flagged': int(r['flagged']),
            'rate': float(r['flagged']) / int(r['n']), 'is_rare': rare.get(r['family'], False)} for r in rows]
    return sorted(out, key=lambda d: (not d['is_rare'], -d['n']))


def make_entry(name, family_name, vh, th, aucs, importance=None, train_s=None, by_family=None,
               fixed_k=None, is_baseline=False, note=None):
    neg_v, pos_v = vh
    neg_t, pos_t = th
    k = fixed_k if fixed_k is not None else best_threshold_k(neg_v, pos_v, 'mcc')
    tp, fp, fn, tn = hist_counts(neg_t, pos_t)
    m = binary_metrics(tp[k], fp[k], fn[k], tn[k])
    m.update(aucs)
    k5 = NBINS // 2
    return {
        'name': name, 'model': family_name, 'note': note, 'is_baseline': bool(is_baseline), 'is_best': False,
        'threshold': k / NBINS,
        'metrics': m,
        'roc_auc': aucs['roc_auc'], 'pr_auc': aucs['pr_auc'],
        'confusion_counts': {'tp': int(tp[k]), 'fp': int(fp[k]), 'fn': int(fn[k]), 'tn': int(tn[k])},
        'fixed_05': binary_metrics(tp[k5], fp[k5], fn[k5], tn[k5]),
        'sweep': sweep_block(neg_t, pos_t),
        'curves': curves_block(neg_t, pos_t),
        'feature_importance': (importance or [])[:25],
        'by_family': by_family or [],
        'train_seconds': train_s,
    }


def lr_importance(model):
    imp = np.abs(model.coefficients.toArray()) * STD
    imp = imp / imp.sum() if imp.sum() else imp
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(FEATURES, imp)], key=lambda d: -d['value'])


rf_kw = dict(featuresCol='features', labelCol='label', numTrees=60, maxDepth=12, maxBins=64,
             subsamplingRate=0.5, minInstancesPerNode=2, featureSubsetStrategy='sqrt', seed=SEED)
SPECS = [
    ('RF · без коррекции',   'RandomForest',      train,       RandomForestClassifier(**rf_kw),
     'естественное соотношение классов'),
    ('RF · веса классов',    'RandomForest',      train,       RandomForestClassifier(weightCol='w', **rf_kw),
     f'вес редкого класса = {W_POS:,.0f}'),
    ('RF · undersampling',   'RandomForest',      train_under, RandomForestClassifier(**rf_kw),
     f'нормальный трафик прорежен до {UNDER_RATIO}:1'),
    ('GBT · веса классов',   'GradientBoosting',  train,
     GBTClassifier(featuresCol='features', labelCol='label', weightCol='w', maxIter=40, maxDepth=5, stepSize=0.2,
                   subsamplingRate=0.7, maxBins=64, seed=SEED),
     f'вес редкого класса = {W_POS:,.0f}'),
    ('LogReg · веса классов', 'LogisticRegression', train,
     LogisticRegression(featuresCol='features', labelCol='label', weightCol='w', maxIter=60, regParam=1e-3, standardization=True),
     'линейная база с весами'),
]

entries, fitted = [], {}
for name, fam_name, tr, est, note in SPECS:
    with stage(f'обучение · {name}'):
        model = est.fit(tr)
    train_s = TIMINGS[f'обучение · {name}']
    with stage(f'оценка · {name}'):
        val_s = score_sdf(model, val).persist()
        test_s = score_sdf(model, test, extra=('family',)).persist()
        vh, th = score_histogram(val_s), score_histogram(test_s)
        aucs = spark_auc(test_s)
        k_star = best_threshold_k(vh[0], vh[1], 'mcc')
        by_family = family_table(test_s, k_star)
        val_s.unpersist(); test_s.unpersist()
    imp = lr_importance(model) if fam_name == 'LogisticRegression' else tree_importance(model, FEATURES)
    entries.append(make_entry(name, fam_name, vh, th, aucs, imp, train_s, by_family, note=note))
    fitted[name] = model
    e = entries[-1]
    print(f'   {name}: thr={e["threshold"]:.3f}  MCC={e["metrics"]["mcc"]:.4f}  PR-AUC={e["pr_auc"]:.4f}  '
          f'recall={e["metrics"]["recall_pos"]:.3f}  precision={e["metrics"]["precision_pos"]:.3f}  FP={e["confusion_counts"]["fp"]}')

# baseline «ничего не ловим»
ref_neg, ref_pos = th
neg0, pos0 = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
neg0[0], pos0[0] = ref_neg.sum(), ref_pos.sum()
PREV_TEST = float(ref_pos.sum() / (ref_pos.sum() + ref_neg.sum()))
entries.append(make_entry('Baseline · всё нормальное', 'Baseline', (neg0, pos0), (neg0, pos0),
                          {'roc_auc': 0.5, 'pr_auc': PREV_TEST}, fixed_k=NBINS // 2, is_baseline=True,
                          note='предсказывает «не атака» для всех потоков'))

trained = [e for e in entries if not e['is_baseline']]
best = max(trained, key=lambda e: e['metrics']['mcc'])
best['is_best'] = True
print('\nЛучшая по MCC:', best['name'], f'MCC={best["metrics"]["mcc"]:.4f}')

## 7. Сводка и графики

Смотрим одновременно accuracy, ROC-AUC, PR-AUC и MCC: на таком дисбалансе первые две почти не различают модели,
а две последние — различают. Отдельно — MCC при пороге 0.5 против настроенного порога.

In [ ]:
tbl = pd.DataFrame([{
    'model': e['name'], 'thr': round(e['threshold'], 3), 'MCC': round(e['metrics']['mcc'], 4),
    'MCC@0.5': round(e['fixed_05']['mcc'], 4), 'PR-AUC': round(e['pr_auc'], 4), 'ROC-AUC': round(e['roc_auc'], 4),
    'recall': round(e['metrics']['recall_pos'], 3), 'precision': round(e['metrics']['precision_pos'], 3),
    'FP': e['confusion_counts']['fp'], 'FN': e['confusion_counts']['fn'], 'accuracy': round(e['metrics']['accuracy'], 5),
} for e in entries]).set_index('model')
display(tbl)
tbl.to_csv(ART_DIR / 'metrics_all_models.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for e in trained:
    axes[0].plot(e['curves']['roc']['fpr'], e['curves']['roc']['tpr'], lw=2, label=f'{e["name"]} · {e["roc_auc"]:.3f}')
    axes[1].plot(e['curves']['pr']['recall'], e['curves']['pr']['precision'], lw=2, label=f'{e["name"]} · {e["pr_auc"]:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=.4); axes[0].set_title('ROC — все модели выглядят хорошо'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(fontsize=7)
axes[1].axhline(PREV_TEST, color='k', ls='--', alpha=.5); axes[1].set_title(f'PR — разница видна (baseline = {PREV_TEST:.4f})'); axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision'); axes[1].legend(fontsize=7)
bf = pd.DataFrame(best['by_family']); bf = bf[(bf.is_rare) | (bf.n > bf.n.max() * 0.0005)].head(14)
axes[2].barh(bf.family, bf.rate, color=['#f43f5e' if r else '#94a3b8' for r in bf.is_rare]); axes[2].invert_yaxis()
axes[2].set_title(f'{best["name"]}: доля помеченных как атака\n(красные — редкие: recall; серые — ложные тревоги)'); axes[2].set_xlim(0, 1)
plt.tight_layout(); plt.savefig(ART_DIR / 'roc_vs_pr.png', bbox_inches='tight'); plt.show()

## 8. Сохранение: `report.json`, `index.json`, модель

Фронт читает `artifacts/classification/network_intrusion/report.json`. NaN/Inf → `null`. Все числа в тексте вывода подставляются из расчёта.

In [ ]:
def pct_(x, d=1):
    return f'{x * 100:.{d}f}%'


naive = next(e for e in entries if e['is_baseline'])
by_roc = max(trained, key=lambda e: e['roc_auc'])
by_pr = max(trained, key=lambda e: e['pr_auc'])
cc = best['confusion_counts']
bm = best['metrics']
rare_fams = [f for f in best['by_family'] if f['is_rare'] and f['n'] >= 1]
best_fam = max(rare_fams, key=lambda f: f['rate']) if rare_fams else None
worst_fam = min(rare_fams, key=lambda f: f['rate']) if rare_fams else None
acc_spread = max(e['metrics']['accuracy'] for e in trained) - min(e['metrics']['accuracy'] for e in trained)
mcc_spread = max(e['metrics']['mcc'] for e in trained) - min(e['metrics']['mcc'] for e in trained)
mcc_by_name = {e['name']: e['metrics']['mcc'] for e in trained}

SPARK_BLOCK = {**SPARK_INFO, 'n_rows_raw': int(N_RAW), 'n_rows_used': int(N_USED), 'timings': TIMINGS,
               'ui': {'master': ':8085', 'history': ':18080'}}

narrative = {
    'context': (f'CICIDS2017: {N_RAW:,} сетевых потоков, после удаления точных дубликатов {N_USED:,}. Положительный класс — редкие атаки '
                f'(Bot, Web Attack, Infiltration, Heartbleed): {N_POS_ALL:,} потоков, {pct_(PREVALENCE, 3)} данных, дисбаланс ≈ 1 : {IMBALANCE:,.0f}. '
                'Остальной трафик включает и «громкие» DoS/DDoS/PortScan — они заметны без модели и не должны давать ложных тревог.'),
    'hero_metric': 'MCC (Matthews Correlation Coefficient)',
    'why_hero': ('MCC использует все четыре клетки confusion matrix и не «выигрывает» на огромном TN. '
                 'Порог каждой модели выбран на валидации именно по MCC.'),
    'why_not_accuracy': (f'Baseline «ничего не ловим» даёт accuracy {pct_(naive["metrics"]["accuracy"], 3)} при recall 0. '
                         f'Разброс accuracy между обученными моделями — всего {acc_spread:.4f}, разброс MCC — {mcc_spread:.3f}: '
                         'accuracy почти не различает модели, MCC — различает.'),
    'why_not_roc_auc': (f'Лучшая по ROC-AUC — {by_roc["name"]} ({by_roc["roc_auc"]:.4f}), лучшая по PR-AUC — {by_pr["name"]} ({by_pr["pr_auc"]:.4f}). '
                        'ROC-кривая раздута огромным числом верно отпущенных потоков, PR-кривая строится только по редкому классу.'),
    'imbalance_methods': ('MCC по способам: ' + ' · '.join(f'{n} = {v:.3f}' for n, v in mcc_by_name.items()) +
                          '. Веса и undersampling сдвигают вероятности, поэтому порог 0.5 для них неприменим — см. «MCC@0.5» в таблице.'),
    'main_conclusion': (f'Лучшая по MCC — {best["name"]}: MCC={bm["mcc"]:.4f}, PR-AUC={best["pr_auc"]:.4f} (baseline по prevalence {PREV_TEST:.5f}), '
                        f'порог {best["threshold"]:.3f}. На тесте поймано {cc["tp"]:,} редких атак, пропущено {cc["fn"]:,}, ложных тревог {cc["fp"]:,} '
                        f'на {cc["tn"] + cc["fp"]:,} обычных потоков.' +
                        (f' Лучше всего ловятся «{best_fam["family"]}» ({pct_(best_fam["rate"])}), хуже всего — «{worst_fam["family"]}» ({pct_(worst_fam["rate"])}).' if (best_fam and worst_fam and best_fam is not worst_fam) else '')),
    'rules': {
        'hero_mcc': 'Hero-метрика MCC; PR-AUC — контрольная; accuracy и ROC-AUC для выбора не используем',
        'threshold_on_val': 'Порог подбирается на валидации, метрики — на тесте',
        'dedup_first': 'Точные дубликаты удаляются до разбиения, иначе утечка train → test',
        'by_family': 'Смотрим recall по семействам атак, а не только итоговый MCC',
    },
}

DIAG = {
    'is_balanced': False,
    'prevalence_positive': float(PREVALENCE), 'imbalance_ratio': float(IMBALANCE),
    'mcc': float(bm['mcc']), 'pr_auc': float(best['pr_auc']), 'roc_auc': float(best['roc_auc']),
    'balanced_accuracy': float(bm['balanced_accuracy']),
    'sensitivity_rare': float(bm['recall_pos']), 'precision_rare': float(bm['precision_pos']),
    'naive_accuracy': float(naive['metrics']['accuracy']), 'naive_mcc': float(naive['metrics']['mcc']),
    'naive_pr_auc': float(naive['pr_auc']),
    'accuracy_spread_models': float(acc_spread), 'mcc_spread_models': float(mcc_spread),
    'pr_auc_over_baseline': float(best['pr_auc'] / max(PREV_TEST, 1e-12)),
    'mcc_at_05': float(best['fixed_05']['mcc']), 'mcc_tuned': float(bm['mcc']),
    'roc_auc_best_name': by_roc['name'], 'pr_auc_best_name': by_pr['name'],
    'tp': cc['tp'], 'fp': cc['fp'], 'fn': cc['fn'], 'tn': cc['tn'],
}

report = {
    'meta': {
        'task': TASK,
        'title': 'Network Intrusion · редкие атаки',
        'subtitle': 'Экстремальный дисбаланс: судим по MCC и PR-AUC, порог — по валидации',
        'dataset': f'CICIDS2017 (Kaggle · {DATASET_NAME})',
        'target': 'редкая атака (Bot, Web Attack, Infiltration, Heartbleed)',
        'classes': ['normal_or_loud', 'rare_attack'], 'positive_class': 'rare_attack',
        'features': FEATURES, 'seed': SEED, 'generated_at': datetime.now().isoformat(),
        'n_samples_raw': int(N_RAW), 'n_samples': int(N_USED), 'n_features': len(FEATURES),
        'n_train': int(N_TRAIN), 'n_val': int(N_VAL), 'n_test': int(N_TEST),
        'target_stats': {
            'prevalence': float(PREVALENCE), 'prevalence_test': PREV_TEST,
            'prevalence_by_fold': {f: v['prevalence'] for f, v in FOLDS.items()},
            'imbalance_ratio': float(IMBALANCE), 'n_positive': int(N_POS_ALL),
        },
        'imbalance': {'w_pos': float(W_POS), 'under_ratio': UNDER_RATIO, 'train_under_rows': int(N_UNDER)},
        'split_note': 'Случайное разбиение 70/10/20 после удаления точных дубликатов; fold зафиксирован в parquet',
        'spark': SPARK_BLOCK,
        'narrative': narrative,
    },
    'headline': {
        'best_model': best['name'], 'best_metric': 'mcc', 'best_metric_value': float(bm['mcc']),
        'verdict': (f'Лучшая по MCC — {best["name"]} · MCC={bm["mcc"]:.4f} · PR-AUC={best["pr_auc"]:.4f} · '
                    f'recall={bm["recall_pos"]:.4f} · FP={cc["fp"]:,}'),
        'diagnostic_verdict': DIAG,
    },
    'models': sorted(entries, key=lambda e: (e['is_baseline'], -e['metrics']['mcc'])),
    'eda': {'families': FAMILIES, 'folds': FOLDS},
    'charts': {k.replace('.png', ''): k for k in ('eda.png', 'roc_vs_pr.png') if (ART_DIR / k).exists()},
}

slug = re.sub(r'[^a-z0-9]+', '_', best['name'].lower()).strip('_')
spark_model_path = MODEL_DIR / f'network_intrusion_{slug}_spark'
try:
    fitted[best['name']].write().overwrite().save(spark_model_path.as_uri())
    saved_model = str(spark_model_path.relative_to(PROJECT))
except Exception as e:
    print('⚠ модель Spark не сохранена:', type(e).__name__, str(e)[:200])
    saved_model = None

meta_path = MODEL_DIR / f'network_intrusion_{slug}_{datetime.now():%Y%m%d_%H%M}.json'
dump_json({'task': TASK, 'model': best['name'], 'features': FEATURES, 'threshold': best['threshold'],
           'metrics': bm, 'saved_at': datetime.now().isoformat(), 'spark_model': saved_model}, meta_path)

report['artifacts'] = {'model_spark': saved_model, 'model_meta': str(meta_path.relative_to(PROJECT)),
                       'metrics_csv': str((ART_DIR / 'metrics_all_models.csv').relative_to(PROJECT))}
report_path = ART_DIR / 'report.json'
dump_json(report, report_path)


def upsert_index(entry):
    p = PROJECT / 'artifacts' / 'index.json'
    idx = json.loads(p.read_text()) if p.exists() else {'generated_at': None, 'tasks': []}
    idx['tasks'] = sorted([t for t in idx['tasks'] if t['id'] != entry['id']] + [entry], key=lambda t: t['id'])
    idx['generated_at'] = datetime.now().isoformat()
    dump_json(idx, p)


upsert_index({'id': TASK, 'title': report['meta']['title'], 'best_model': best['name'], 'hero_metric': 'mcc',
              'hero_value': float(bm['mcc']), 'unit': '', 'report': str(report_path.relative_to(PROJECT)),
              'charts_dir': str(ART_DIR.relative_to(PROJECT))})

print('report →', report_path, f'({report_path.stat().st_size / 1024:.0f} КБ)')
print(report['headline']['verdict'])
print('время этапов:', json.dumps(TIMINGS, ensure_ascii=False))

## 9. Итог

* Данные: ~2.8 млн реальных сетевых потоков; весь пайплайн (чтение, очистка, дедупликация, обучение пяти моделей, оценка) выполнен на Spark.
* Метрики выбираются **структурой данных**: при доле редкого класса в доли процента accuracy и ROC-AUC почти не различают модели, MCC и PR-AUC — различают.
* Способ борьбы с дисбалансом меняет не только качество, но и **шкалу вероятностей** — поэтому порог настраивается на валидации, а «0.5» показан лишь для контраста.
* Итоговый MCC нужно читать вместе с разбором по семействам: модель может хорошо ловить Bot и совсем не видеть редкие веб-атаки.

**Артефакты:** `artifacts/classification/network_intrusion/{report.json, eda.png, roc_vs_pr.png, metrics_all_models.csv}`.

Остановить Spark: `spark.stop()`.